# Snowpark Notebook: Data Quality Checks

This notebook performs row count, null-value, and duplicate checks across migrated Snowflake tables.

In [ ]:
import os
import pandas as pd
from typing import Optional
from snowflake.snowpark import Session


def _env(name: str, default: Optional[str] = None) -> str:
    value = os.getenv(name, default)
    if value is None:
        raise EnvironmentError(f"Missing required environment variable: {name}")
    return value


connection_parameters = {
    'account': _env('SNOWFLAKE_ACCOUNT'),
    'user': _env('SNOWFLAKE_USER'),
    'password': _env('SNOWFLAKE_PASSWORD'),
    'role': _env('SNOWFLAKE_ROLE', 'ACCOUNTADMIN'),
    'warehouse': _env('SNOWFLAKE_WAREHOUSE', 'ADVENTUREWORKS_ETL_WH'),
    'database': _env('SNOWFLAKE_DATABASE', 'ADVENTUREWORKS_MIGRATED'),
}

DATABASE = connection_parameters['database']
STAGING_SCHEMA = 'STAGING'
DW_SCHEMA = 'DW'

session = Session.builder.configs(connection_parameters).create()
session.sql('SELECT CURRENT_ACCOUNT() AS account, CURRENT_ROLE() AS role').show()


In [ ]:
def quote_ident(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'


def fqtn(schema: str, table: str) -> str:
    return f"{quote_ident(DATABASE)}.{quote_ident(schema)}.{quote_ident(table)}"


def table_columns(schema: str, table: str) -> list[str]:
    sql = f"""
      SELECT COLUMN_NAME
      FROM {quote_ident(DATABASE)}.INFORMATION_SCHEMA.COLUMNS
      WHERE TABLE_SCHEMA = '{schema.upper()}'
        AND TABLE_NAME = '{table.upper()}'
      ORDER BY ORDINAL_POSITION
    """
    return [row['COLUMN_NAME'] for row in session.sql(sql).collect()]


staging_tables = [
    row['TABLE_NAME']
    for row in session.sql(
        f"""
        SELECT TABLE_NAME
        FROM {quote_ident(DATABASE)}.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = '{STAGING_SCHEMA}'
          AND TABLE_TYPE = 'BASE TABLE'
        ORDER BY TABLE_NAME
        """
    ).collect()
]

dw_tables = {
    row['TABLE_NAME']
    for row in session.sql(
        f"""
        SELECT TABLE_NAME
        FROM {quote_ident(DATABASE)}.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = '{DW_SCHEMA}'
          AND TABLE_TYPE = 'BASE TABLE'
        """
    ).collect()
}

special_dw_targets = {
    'PROSPECTIVE_BUYER_STG': 'DIM_PROSPECTIVE_BUYER',
    'NEW_FACT_CURRENCY_RATE_STG': 'FACT_CURRENCY_RATE_INCREMENTAL',
}

row_count_pairs = []
for staging_table in staging_tables:
    dw_table = special_dw_targets.get(staging_table.upper(), staging_table[:-4] if staging_table.upper().endswith('_STG') else staging_table)
    if dw_table.upper() in dw_tables:
        row_count_pairs.append((staging_table, dw_table))

pd.DataFrame(row_count_pairs, columns=['staging_table', 'dw_table'])


In [ ]:
row_count_results = []
for staging_table, dw_table in row_count_pairs:
    staging_count = session.sql(f"SELECT COUNT(*) AS CNT FROM {fqtn(STAGING_SCHEMA, staging_table)}").collect()[0]['CNT']
    dw_count = session.sql(f"SELECT COUNT(*) AS CNT FROM {fqtn(DW_SCHEMA, dw_table)}").collect()[0]['CNT']
    row_count_results.append(
        {
            'staging_table': staging_table,
            'dw_table': dw_table,
            'staging_count': staging_count,
            'dw_count': dw_count,
            'difference': dw_count - staging_count,
            'status': 'PASS' if dw_count == staging_count else 'FAIL',
        }
    )

row_count_df = pd.DataFrame(row_count_results).sort_values('staging_table')
row_count_df


In [ ]:
null_checks = []
not_null_columns = session.sql(
    f"""
    SELECT TABLE_NAME, COLUMN_NAME
    FROM {quote_ident(DATABASE)}.INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = '{DW_SCHEMA}'
      AND IS_NULLABLE = 'NO'
    ORDER BY TABLE_NAME, ORDINAL_POSITION
    """
).collect()

for row in not_null_columns:
    table_name = row['TABLE_NAME']
    column_name = row['COLUMN_NAME']
    sql = f"SELECT COUNT(*) AS CNT FROM {fqtn(DW_SCHEMA, table_name)} WHERE {quote_ident(column_name)} IS NULL"
    null_count = session.sql(sql).collect()[0]['CNT']
    null_checks.append(
        {
            'table_name': table_name,
            'column_name': column_name,
            'null_count': null_count,
            'status': 'PASS' if null_count == 0 else 'FAIL',
        }
    )

null_check_df = pd.DataFrame(null_checks)
null_check_df


In [ ]:
duplicate_checks = []
for table_name in sorted(dw_tables):
    columns = table_columns(DW_SCHEMA, table_name)
    key_candidates = [c for c in columns if c.upper().endswith('KEY')]
    if not key_candidates:
        continue

    key_col = key_candidates[0]
    has_is_current = '_IS_CURRENT' in {c.upper() for c in columns}
    where_clause = 'WHERE _is_current = TRUE' if has_is_current else ''

    duplicate_sql = f"""
      SELECT
        COUNT(*) AS duplicate_groups,
        COALESCE(SUM(record_count), 0) AS duplicate_rows
      FROM (
        SELECT {quote_ident(key_col)} AS key_value, COUNT(*) AS record_count
        FROM {fqtn(DW_SCHEMA, table_name)}
        {where_clause}
        GROUP BY {quote_ident(key_col)}
        HAVING COUNT(*) > 1
      ) d
    """

    metrics = session.sql(duplicate_sql).collect()[0]
    duplicate_checks.append(
        {
            'table_name': table_name,
            'key_column': key_col,
            'duplicate_groups': metrics['DUPLICATE_GROUPS'],
            'duplicate_rows': metrics['DUPLICATE_ROWS'],
            'status': 'PASS' if metrics['DUPLICATE_GROUPS'] == 0 else 'FAIL',
        }
    )

duplicate_df = pd.DataFrame(duplicate_checks)
duplicate_df


In [ ]:
summary = {
    'row_count_pass': int((row_count_df['status'] == 'PASS').sum()),
    'row_count_fail': int((row_count_df['status'] == 'FAIL').sum()),
    'null_pass': int((null_check_df['status'] == 'PASS').sum()),
    'null_fail': int((null_check_df['status'] == 'FAIL').sum()),
    'duplicate_pass': int((duplicate_df['status'] == 'PASS').sum()),
    'duplicate_fail': int((duplicate_df['status'] == 'FAIL').sum()),
}

pd.DataFrame([summary])


In [ ]:
session.close()
print('Data quality notebook execution completed.')
